In [1]:
import xarray as xr
import numpy as np
import numpy.ma as ma
from datetime import datetime as dt
from datetime import timedelta

import matplotlib.pyplot as plt
import cartopy.crs as ccrs

import glob, sys, os, re
from pprint import pprint
import subprocess

In [2]:
sim_dir = "/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km"
fig_dir = '../figures'

In [3]:
# Define simulation subfolders
sim_names = [f for f in os.listdir(sim_dir) if re.match("(.*)2016(.*)",f) ]
sim_names.sort(reverse=True)
print(sim_names)

['t0_20160910_00', 'p6_20160910_06', 'p12_20160910_12']
The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [5]:
# Input time info
time_freq_MNH = 1200 # s
time_step_MNH = 3 # s

# Simulation lag info
lags_h = np.array([0,6,12])
lag_6h = timedelta(seconds=6*3600)
lag_12h = timedelta(seconds=12*3600)

# Conversion factors
s_to_h = 1/3600
rho_w = 1e3 # km/m3
m_to_mm = 1e3 

In [7]:
# Load all sims at a specific time

def getFileAtTime(target_time,time_ref=dt(2016,9,10),lag_h=0):
    """For a given lag (in hours) at the start of the simulation,
    get the filename of the right simulation at that given time.
    """
    
    i_lag = np.where(lags_h == lag_h)[0][0]
    
    #-- get corresponding time file
    # time difference
    delta_t = target_time - (time_ref+timedelta(seconds=lag_h*3600))
    # find corresponding subfolder
    i_seg = int((delta_t.total_seconds()/3600/Dt - 1) / (12/Dt) ) +1 # segment number
    subfolder = "seg0%d_12h_tege"%i_seg
    # build corresponding filename
    i_t_seg = int((delta_t.total_seconds()/3600/Dt - 1) % int(12/Dt) + 1)
    filename = "FBC01.1.MNH0%d.OUT.%s.nc"%(i_seg,str(i_t_seg).zfill(3))
    # full path
    path = os.path.join(sim_dir,sim_names[i_lag],subfolder,filename)

    return path

#- MNH File identification

def getMNHFileAtTime(target_time,time_ref=dt(2016,9,10),lag_h=0):
    """For a given lag (in hours) at the start of the simulation,
    get the filename of the right simulation at that given time.
    """
    
    # simulation index
    i_lag = np.where(lags_h == lag_h)[0][0]
    # simulation output hourly frequency
    Dt = time_freq_MNH/3600
    
    #-- get corresponding time file
    # time difference
    delta_t = target_time - (time_ref+timedelta(seconds=lag_h*3600))
    # find corresponding subfolder
    i_seg = int((delta_t.total_seconds()/3600/Dt - 1) / (12/Dt) ) +1 # segment number
    subfolder = "seg0%d_12h_tege"%i_seg
    # build corresponding filename
    i_t_seg = int((delta_t.total_seconds()/3600/Dt - 1) % int(12/Dt) + 1)
    filename = "FBC01.1.MNH0%d.OUT.%s.nc"%(i_seg,str(i_t_seg).zfill(3))
    # full path
    path = os.path.join(sim_dir,sim_names[i_lag],subfolder,filename)

    return path

#- Adjust target time for file selection

def getAdjustedTime(target_time,time_freq=1800):
    """Adjust the target datetime to the current time, according to the frequency of data files.

    Args:
    - target_time (datetime object): target time
    - time_freq (float): frequency of the data files, in s

    Returns:
    - adjusted time (datetime object): adjusted time on the data files time array.
    """

    # compute the number of seconds of IMERG file since start of day
    n_steps = int((target_time-dt(target_time.year,target_time.month,target_time.day)).seconds / (time_freq))
    adjusted_seconds = int(n_steps * time_freq)
    # adjusted time
    adjusted_time = dt(target_time.year,target_time.month,target_time.day)+timedelta(seconds=adjusted_seconds)

    return adjusted_time

In [ ]:
# time to show
target_time = dt(2016,9,10,14,00)
# Adjusted time on MNH time array
target_time_MNH = getAdjustedTime(target_time,time_freq_MNH)

In [6]:
##-- Get Meso-NH data and coords

#- Datasets
data_lag0 = xr.open_dataset(getMNHFileAtTime(target_time_MNH,lag_h=0))
data_lag6 = xr.open_dataset(getMNHFileAtTime(target_time_MNH,lag_h=6))
data_lag12 = xr.open_dataset(getMNHFileAtTime(target_time_MNH,lag_h=12))

#- Domain coords
lon_array = data_lag0.longitude.values
lat_array = data_lag0.latitude.values
longitude = data_lag0.longitude[0].values
latitude = data_lag0.latitude[:,0].values
extent = [longitude[0],longitude[-1],latitude[0],latitude[-1]]

NameError: name 'getMNHFileAtTime' is not defined

In [ ]:
##